For each NACE Class get the 100 chunks that scored highest across all the reports 

In [42]:
import pandas as pd
import glob
import os
import tqdm
import numpy as np
import matplotlib.pyplot as plt
os.chdir("/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis")
from test_base import *

In [43]:
os.chdir('/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis')

In [44]:
overview_path = "data/datasets/stoxx_600/stoxx_600_overview.csv"
overview_path = "data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"
overview_path = "data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_overview.csv"

In [45]:
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_0_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/tables_cos_sim_0.0_nace_level_1_stoxx/"
raw_data_path = "results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_6_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "results/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"

In [46]:
reports = glob.glob(raw_data_path + "*/*_short.csv")
reports = glob.glob(raw_data_path + "*/*_long.csv")
len(reports)

3053

In [31]:
#sample_ratio = 1

In [32]:
#max_elements_per_class = 1000000
#top_k_sentences = 200000

In [ ]:
# if true, adds only chunks to training that have been classified into the same NACE class its report comes from
#filter_only_right_chunks = True

In [ ]:
# if true, adds random chunks that do not fulfill the minimum treshold (for BERT Training)
#with_null_classifiers = True

In [47]:
new_threshold_cos_sin = 0.4

In [48]:
nace_level_descriptions = 1
nace_level = 1
assert nace_level_descriptions >= nace_level

In [49]:
training_data_path = "data/training_data"

In [50]:
#suffix = f"sample_ratio_{sample_ratio}" + ("__filter_only_right_chunks" if filter_only_right_chunks else "") + ("__with_null_classifiers" if with_null_classifiers else "") + f"__nace_level_{nace_level}"
suffix = f"2nd_approach" + f"__nace_level_{nace_level}__cos_thres_{new_threshold_cos_sin}"

end_path = os.path.join(training_data_path, 
                        raw_data_path.split("/")[-2] + "__" + suffix)
end_path
os.makedirs(end_path, exist_ok=True)

In [52]:
#df_overview = pd.read_excel(overview_path)
df_overview = pd.read_csv(overview_path, index_col=0)
df_overview.head()

,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,NACE,...,ISIN,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report
0,CA05335P1099,Auxly Cannabis Group Inc.,1,1987.0,CAN,05335P109,19960301.0,CAN,003V9K-E,1.30,...,CA05335P1099,BDGMQB,Auxly Cannabis Group Inc.,1.0,XLY-CA,1,SHARE,BDGMQB3,A,Auxly Cannabis Group Inc.2.pdf
1,JP3947800003,"MEGMILK SNOW BRAND Co., Ltd.",1,2009.0,JPN,J41966102,20220727.0,JPN,0833Y1-E,1.41,...,JP3947800003,B3ZC07,"MEGMILK SNOW BRAND Co., Ltd.",1.0,MMSBF-US,0,SHARE,BKQN701,A,"MEGMILK SNOW BRAND Co., Ltd.2.pdf"
2,ID1000167901,PT Cilacap Samudera Fishing Industry Tbk,0,1999.0,IDN,Y129H6108,20220527.0,IDN,@NA,3.12,...,ID1000167901,BMBMZG,PT Cilacap Samudera Fishing Industry Tbk,1.0,ASHA-ID,0,SHARE,BMBMZG9,A,PT Cilacap Samudera Fishing Industry Tbk3.pdf
3,JP3843250006,Hokuto Corporation,1,1964.0,JPN,J2224T102,19941202.0,JPN,05HY7N-E,1.30,...,JP3843250006,643271,Hokuto Corporation,1.0,1379-JP,1,SHARE,6432715,A,Hokuto Corporation1.pdf
4,VN000000VTQ6,Viet Trung Quang Binh Joint Stock Co,0,1961.0,VNM,Y937XS108,NaN,VNM,@NA,2.30,...,VN000000VTQ6,BMCR2W,Viet Trung Quang Binh Joint Stock Co,1.0,VTQ-VN,0,SHARE,BMCR2W8,A,Viet Trung Quang Binh Joint Stock Co3.pdf


In [53]:
df_nace_codes_descriptions = pd.read_csv("data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
filter_level_1_classes = "ABCDEFGHIJKLMNOPQRSTUVW"

For each class c: those paragraphs p of reports in class c with cos-sim(p, c) > 0.5

In [54]:
result = pd.DataFrame(columns=["Sentences", "Score", "NACE_Code"])

# loop over all reports and get the all the sentences 
for report in tqdm.tqdm(reports):

    df = pd.read_csv(report)
    
    report_name = os.path.basename(report).replace(".txt_long.csv", "") + ".pdf"
    report_code = df_overview[df_overview["Report"]==report_name]["NACE"].iloc[0]
    report_code = get_all_level(report_code)[nace_level]

    scores = [c for c in df.columns if "Scores" in c]
    
    df["max_class_sim"] = [scores[i][7] for i in np.argmax(df[scores], 1)]
    df["Score"] = df[scores].max(1)
    
    df.loc[(df["max_class_sim"] == report_code) & (df["Score"] > new_threshold_cos_sin),"NACE_Code"] = report_code
    df["NACE_Code"] = df["NACE_Code"].fillna("NO_CLASS")

    result = pd.concat([result, df[["Sentences", "Score", "NACE_Code"]]])


  0%|                                                                                                                                                                                        | 0/3053 [00:00<?, ?it/s]/tmp/ipykernel_190914/640071683.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, df[["Sentences", "Score", "NACE_Code"]]])
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3053/3053 [02:55<00:00, 17.44it/s]


In [55]:
stats = result.groupby("NACE_Code").count()["Sentences"]
stats

NACE_Code
A              1372
B              9239
C              1678
D             13093
E              4852
F             10094
G              6264
H              8845
I              2758
J              1592
K             35927
L              9569
M              2074
N              1660
NO_CLASS    2444776
P              1605
Q               858
R               342
S               118
T                21
Name: Sentences, dtype: int64

In [56]:
amount_no_class = stats[stats.index != "NO_CLASS"].max().item()
amount_no_class

35927

In [57]:
result_right = result[result["NACE_Code"] != "NO_CLASS"]
result_NO_CLASS = result.loc[result["NACE_Code"] == "NO_CLASS"].sample(n=amount_no_class)
result_final = pd.concat([result_right, result_NO_CLASS], axis=0)

In [58]:
stats = result_final.groupby("NACE_Code").agg({
    "Sentences": "count", 
    "Score": "mean"
})
stats

,Sentences,Score
NACE_Code,,
A,1372,0.442954
B,9239,0.460934
C,1678,0.452970
D,13093,0.458277
E,4852,0.478435
F,10094,0.482483
G,6264,0.446090
H,8845,0.461036
I,2758,0.448902


In [59]:
stats.to_csv(end_path + "/statistics.csv")

In [60]:
# recordings.append({"Code": code,"Nbr. of Chunks": len(temp),"Avg. Length": temp["Sentences"].apply(len).mean(), "Avg. Score": temp["Score"].mean(), "Min. Score": temp["Score"].min(), "Max. Score": temp["Score"].max()})

# df_recordings = pd.DataFrame(recordings)
# df_recordings = df_recordings.sort_values(by="Code")
# df_recordings.head()

#df_recordings.to_csv(end_path + "/statistics.csv")

In [61]:
full_df= result_final.rename(columns={"Sentences": "text"})
#full_df = full_df.drop(columns="Score")
full_df["Evaluation"] = None
full_df["Notes"] = None

In [62]:
# Store each class for reading
n = 40
for nace_class in stats.index: 
    print(nace_class)
    temp = full_df[full_df["NACE_Code"] == nace_class].copy()
    temp["Evaluation"] = None
    temp["Notes"] = None
    if len(temp) >= n:
        temp = temp.sample(n=40)
    temp.to_csv(os.path.join(end_path, nace_class + ".csv"))

A
B
C
D
E
F
G
H
I
J
K
L
M
N
NO_CLASS
P
Q
R
S
T


In [63]:
from sklearn.model_selection import train_test_split

# Split full_df into train (60%) and temp (40%)
train_df, temp_df = train_test_split(full_df, test_size=0.4, random_state=42)

# Split temp into test (20%) and validation (20%)
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print the sizes of each split
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}, Validation size: {len(val_df)}")

Train size: 88732, Test size: 29578, Validation size: 29578


In [64]:
full_df.to_csv(end_path + "/full_data.csv", index=False)

In [65]:
train_df.to_csv(end_path + "/train_data.csv", index=False)
val_df.to_csv(end_path + "/val_data.csv", index=False)
test_df.to_csv(end_path + "/test_data.csv", index=False)

In [66]:
end_path

'data/training_data/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__2nd_approach__nace_level_1__cos_thres_0.4'